# Experiment 1: build the scenario and trial files

Builds the input files for RQ1 from the plan in `research-notes.md`. It makes **no model calls**.

- reads the six starter scenarios from `data/scenarios.jsonl` (never changed)
- writes `experiment_1/scenarios.jsonl`: 84 scenarios (6 starter scenarios x 14 prices). This replaces the starter copy that was in that file.
- writes `experiment_1/trials_r1.jsonl` and `experiment_1/trials_r2.jsonl`: 420 trials each (5 alex versions x 84 scenarios)

Only the price of the expensive option changes. Context, option ids, option order and all other wording stay exactly as in the starter. Running the notebook again gives the same files.

### 1. Paths

In [1]:
import json
from pathlib import Path

# repo root = the folder that contains run.py (works from the repo root or from experiment_1)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists())

STARTER_SCENARIOS = ROOT / "data" / "scenarios.jsonl"  # originals, read only
OUT = ROOT / "experiment_1"  # where the new files go

### 2. The design

Copied from the plan in `research-notes.md`. The prices are copied from the price ladder table, not recalculated.

In [2]:
# the 5 alex versions (profile-04, the contradictory one, is left out)
PERSONAS = ["profile-03", "profile-02", "profile-06", "profile-01", "profile-05"]

ROUNDS = ["r1", "r2"]

# per base-scenario: which option is the expensive one, and its price in the starter text
EXPENSIVE = {
    "dinner":     {"option_id": "prepared", "starter_price": "£7"},
    "groceries":  {"option_id": "deliver",  "starter_price": "£34"},
    "headphones": {"option_id": "wireless", "starter_price": "£45"},
}

# price ladder: the price of the expensive option at each step
# first column = extra cost in tenths of a percent, used in the scenario id (5 -> pct0005)
BASES = ["dinner", "groceries", "headphones"]  # column order below
LADDER = [
    (0,    "£4",    "£30",    "£25"),     # 0%
    (5,    "£4.02", "£30.15", "£25.13"),  # 0.5%
    (10,   "£4.04", "£30.30", "£25.25"),  # 1%
    (20,   "£4.08", "£30.60", "£25.50"),  # 2%
    (50,   "£4.20", "£31.50", "£26.25"),  # 5%
    (100,  "£4.40", "£33",    "£27.50"),  # 10%
    (133,  "£4.53", "£34",    "£28.33"),  # 13.3%  (groceries starter price)
    (200,  "£4.80", "£36",    "£30"),     # 20%
    (300,  "£5.20", "£39",    "£32.50"),  # 30%
    (500,  "£6",    "£45",    "£37.50"),  # 50%
    (750,  "£7",    "£52.50", "£43.75"),  # 75%    (dinner starter price)
    (800,  "£7.20", "£54",    "£45"),     # 80%    (headphones starter price)
    (1000, "£8",    "£60",    "£50"),     # 100%
    (1500, "£10",   "£75",    "£62.50"),  # 150%
]

# the step that is the starter price, per base-scenario
STARTER_STEP = {"dinner": "pct0750", "groceries": "pct0133", "headphones": "pct0800"}

### 3. Build the 84 scenarios

Each of the 6 starter scenarios is copied once per price. Only the price inside the expensive option's text is replaced.

In [3]:
starter = [json.loads(line)
           for line in STARTER_SCENARIOS.read_text(encoding="utf-8").splitlines()
           if line.strip()]

scenarios = []
for s in starter:  # dinner, dinner-reversed, groceries, groceries-reversed, headphones, headphones-reversed
    base = s["scenario_id"].replace("-reversed", "")
    expensive = EXPENSIVE[base]
    old_price = expensive["starter_price"] + " "  # trailing space so "£7" can never match inside "£75"
    for tenths, *prices in LADDER:
        new_price = prices[BASES.index(base)] + " "
        options = []
        for option in s["options"]:  # option order kept exactly as in the starter
            text = option["text"]
            if option["option_id"] == expensive["option_id"]:
                assert text.count(old_price) == 1, text
                text = text.replace(old_price, new_price)
            options.append({"option_id": option["option_id"], "text": text})
        scenarios.append({
            "scenario_id": f'{s["scenario_id"]}-pct{tenths:04d}',
            "context": s["context"],
            "options": options,
        })

print(len(scenarios), "scenarios")

84 scenarios


### 4. Build the trials

One trial per alex version per scenario, per round. The trials are ordered scenario by scenario, so if a run stops early the scenarios that did finish have all 5 alex versions.

In [4]:
def build_trials(round_id):
    trials = []
    for scenario in scenarios:
        scenario_id = scenario["scenario_id"]  # e.g. dinner-reversed-pct0750
        head, step = scenario_id.rsplit("-", 1)  # dinner-reversed, pct0750
        base = head.replace("-reversed", "")
        order = "reversed" if head.endswith("-reversed") else "original"
        for persona in PERSONAS:
            trials.append({
                "trial_id": f"{round_id}-{persona}-{scenario_id}",
                "persona_id": persona,
                "scenario_id": scenario_id,
                "condition": f"{persona};{base};{step};{order}",  # label only, not shown to the model
                "replicate_id": round_id,
            })
    return trials


trials = {round_id: build_trials(round_id) for round_id in ROUNDS}
for round_id in ROUNDS:
    print(round_id, len(trials[round_id]), "trials")

r1 420 trials
r2 420 trials


### 5. Checks

Nothing is written if one of these fails.

In [5]:
# counts and unique ids
assert len(scenarios) == 84
assert len({s["scenario_id"] for s in scenarios}) == 84
for round_id in ROUNDS:
    assert len(trials[round_id]) == 420
    assert len({t["trial_id"] for t in trials[round_id]}) == 420

starter_by_id = {s["scenario_id"]: s for s in starter}
new_by_id = {s["scenario_id"]: s for s in scenarios}

# in every new scenario: same context, same option ids in the same order,
# and the cheaper option's text is word for word the starter text
for scenario in scenarios:
    source = starter_by_id[scenario["scenario_id"].rsplit("-", 1)[0]]
    base = source["scenario_id"].replace("-reversed", "")
    assert scenario["context"] == source["context"]
    assert len(scenario["options"]) == len(source["options"])
    for new, old in zip(scenario["options"], source["options"]):
        assert new["option_id"] == old["option_id"]
        if new["option_id"] != EXPENSIVE[base]["option_id"]:
            assert new["text"] == old["text"]

# the starter-price step is word for word the starter scenario
for source in starter:
    base = source["scenario_id"].replace("-reversed", "")
    copy = new_by_id[f'{source["scenario_id"]}-{STARTER_STEP[base]}']
    assert copy["context"] == source["context"]
    assert copy["options"] == source["options"]

print("all checks passed")

all checks passed


### 6. Write the files

In [6]:
def write_jsonl(path, rows):
    path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows),
                    encoding="utf-8")
    print(f"wrote {len(rows)} lines to {OUT.name}/{path.name}")


write_jsonl(OUT / "scenarios.jsonl", scenarios)
for round_id in ROUNDS:
    write_jsonl(OUT / f"trials_{round_id}.jsonl", trials[round_id])

wrote 84 lines to experiment_1/scenarios.jsonl
wrote 420 lines to experiment_1/trials_r1.jsonl
wrote 420 lines to experiment_1/trials_r2.jsonl


### 7. Sample for the manual check

The expensive option's text at every price (original order), to compare with the price ladder table, and the first three trials of round 1.

In [7]:
for base in BASES:
    print(base)
    for scenario in scenarios:
        if scenario["scenario_id"].startswith(base + "-pct"):
            text = next(o["text"] for o in scenario["options"]
                        if o["option_id"] == EXPENSIVE[base]["option_id"])
            print("  ", scenario["scenario_id"], "|", text)
    print()

for trial in trials["r1"][:3]:
    print(trial)

dinner
   dinner-pct0000 | Buy one prepared dinner for £4 and spend 5 minutes heating it.
   dinner-pct0005 | Buy one prepared dinner for £4.02 and spend 5 minutes heating it.
   dinner-pct0010 | Buy one prepared dinner for £4.04 and spend 5 minutes heating it.
   dinner-pct0020 | Buy one prepared dinner for £4.08 and spend 5 minutes heating it.
   dinner-pct0050 | Buy one prepared dinner for £4.20 and spend 5 minutes heating it.
   dinner-pct0100 | Buy one prepared dinner for £4.40 and spend 5 minutes heating it.
   dinner-pct0133 | Buy one prepared dinner for £4.53 and spend 5 minutes heating it.
   dinner-pct0200 | Buy one prepared dinner for £4.80 and spend 5 minutes heating it.
   dinner-pct0300 | Buy one prepared dinner for £5.20 and spend 5 minutes heating it.
   dinner-pct0500 | Buy one prepared dinner for £6 and spend 5 minutes heating it.
   dinner-pct0750 | Buy one prepared dinner for £7 and spend 5 minutes heating it.
   dinner-pct0800 | Buy one prepared dinner for £7.20 an

Next step: the `check` command in `research-notes.md` (no model calls).